In [1]:
import numpy as np
import rasterio
from pathlib import Path
from rasterio.windows import from_bounds
from rasterio.warp import reproject, Resampling

tiles_dir = Path(r"C:\Users\leoni\kirgis_repo\segmenteverygrain\leonieexamples\2509_with_vegetation_mask_nodataset\inputtiles")

veg_mask_path = r"C:\Users\leoni\kirgis_repo\veg-shad-wat-filtering\1_uav-filters\vegetation\veg_masks_rgbvi_fallback\25092025_Aktal_gravel_2_10m_OM_RGB_utm_coregistered_ws128_clipwater_vegmask_rgbvi.tif"
shadow_mask_path = r"C:\Users\leoni\kirgis_repo\veg-shad-wat-filtering\1_uav-filters\shadow\0-02\25092025_Aktal_gravel_2_10m_OM_RGB_utm_coregistered_ws128_clipwater_shadowmask.tif"

out_dir = tiles_dir.parent / (tiles_dir.name + "_masked")
out_dir.mkdir(parents=True, exist_ok=True)


def read_mask_aligned(mask_src, dst_src):
    """
    Liest den Masken-Ausschnitt passend zum Tile (dst_src) und bringt ihn
    per nearest-neighbor auf exakt das Tile-Grid (falls nötig).
    """
    # Window im Maskenraster, basierend auf den Bounds des Tiles
    b = dst_src.bounds
    win = from_bounds(b.left, b.bottom, b.right, b.top, transform=mask_src.transform)
    win = win.round_offsets().round_lengths()

    mask_crop = mask_src.read(1, window=win, boundless=True, fill_value=0)

    aligned = np.zeros((dst_src.height, dst_src.width), dtype=np.uint8)

    reproject(
        source=mask_crop,
        destination=aligned,
        src_transform=mask_src.window_transform(win),
        src_crs=mask_src.crs,
        dst_transform=dst_src.transform,
        dst_crs=dst_src.crs,
        resampling=Resampling.nearest,
    )
    return aligned


with rasterio.open(veg_mask_path) as veg_src, rasterio.open(shadow_mask_path) as shad_src:
    for tile_path in sorted(tiles_dir.glob("*.tif")):
        with rasterio.open(tile_path) as src_rgb:
            rgb = src_rgb.read()  # (bands, h, w)
            profile = src_rgb.profile.copy()

            veg_aligned = read_mask_aligned(veg_src, src_rgb)
            shad_aligned = read_mask_aligned(shad_src, src_rgb)

            # robust: alles >0 gilt als "maskiert" (egal ob 1 oder 255)
            invalid = (veg_aligned > 0) | (shad_aligned > 0)

            # NoData Wert: bei uint8 typischerweise 0
            nodata_value = 0
            profile.update(
                nodata=nodata_value,
                compress="deflate",
                tiled=True,
                driver="GTiff",
            )

            # Pixel überschreiben (für Tools, die keine Masken beachten)
            rgb[:, invalid] = nodata_value

            out_path = out_dir / tile_path.name.replace(".tif", "_masked.tif")
            with rasterio.open(out_path, "w", **profile) as dst:
                dst.write(rgb)
                # Zusätzlich eine echte GDAL-Maske schreiben (QGIS liebt das)
                dst.write_mask(np.where(invalid, 0, 255).astype("uint8"))

        print(f"OK: {out_path}")

OK: C:\Users\leoni\kirgis_repo\segmenteverygrain\leonieexamples\2509_with_vegetation_mask_nodataset\inputtiles_masked\tile_r000006_c000024_masked.tif
OK: C:\Users\leoni\kirgis_repo\segmenteverygrain\leonieexamples\2509_with_vegetation_mask_nodataset\inputtiles_masked\tile_r000006_c000025_masked.tif
OK: C:\Users\leoni\kirgis_repo\segmenteverygrain\leonieexamples\2509_with_vegetation_mask_nodataset\inputtiles_masked\tile_r000006_c000028_masked.tif
OK: C:\Users\leoni\kirgis_repo\segmenteverygrain\leonieexamples\2509_with_vegetation_mask_nodataset\inputtiles_masked\tile_r000008_c000015_masked.tif
OK: C:\Users\leoni\kirgis_repo\segmenteverygrain\leonieexamples\2509_with_vegetation_mask_nodataset\inputtiles_masked\tile_r000008_c000026_masked.tif
OK: C:\Users\leoni\kirgis_repo\segmenteverygrain\leonieexamples\2509_with_vegetation_mask_nodataset\inputtiles_masked\tile_r000008_c000031_masked.tif
OK: C:\Users\leoni\kirgis_repo\segmenteverygrain\leonieexamples\2509_with_vegetation_mask_nodataset\